# The corpus against a Bible corpus (eBible)

**Question.** The Bible is the other widely used multi-parallel religious corpus. Would it have served
this study as well, and do the paper's findings hold on it?

**Data.** The eBible corpus (BibleNLP, verse-per-line files aligned to one verse reference list),
downloaded from its public GitHub repository. For each of the corpus's 40 languages, the notebook
lists every eBible translation, keeps those whose text is in the same script as the corpus's, and picks
the one with the most usable verses. A verse is usable if it is non-empty, is not a `<range>` marker,
and is not merged with the next verse. Deuterocanonical books are left out.

**Four questions, decided in advance:**

| | Question | Measured by |
|---|---|---|
| **A** | What does eBible cover of the 40 languages? | languages with a usable translation; full Bible vs NT only vs partial; verses shared by every covered language, against the corpus's 6,236 × 40 with no gaps; publishers and dates |
| **B** | Does the tokenizer audit replicate on shared Bible verses? | per tokenizer: Spearman ρ of language fertility with the corpus; p90/p10 on both; rank agreement of p90/p10 across tokenizers; direction of the Brahmic/Latin tokens-per-byte ratio |
| **C** | How much does translator choice move fertility in each? | within-language spread across translations of the same language, against the corpus's translator study |
| **D** | Does the retrieval floor replicate? (GPU, optional) | English-paired top-1 for the four encoders on 1,000 shared verses; rank agreement with the corpus; which languages are at the bottom in each |

**Runtime.** Parts A–C run on CPU in about 10 minutes. Part D needs a GPU (T4, about 10 minutes); if no
GPU is attached it is skipped and everything else still runs. The setup cell asks for four files **one
at a time** and says where each one is.

## 0. Setup

In [ ]:
!pip -q install tiktoken sentencepiece sentence-transformers
import os, re, io, json, itertools, unicodedata, urllib.request, urllib.error, time
from collections import Counter
import numpy as np, pandas as pd
from scipy import stats

OUT = "results_bible"; os.makedirs(OUT, exist_ok=True)
EBIBLE = os.environ.get("EBIBLE_BASE", "https://raw.githubusercontent.com/BibleNLP/ebible/main")
N_BOOT = 2000
NEED = {
    "fertility_production_v3_spm.csv": "Quran folder, or anon_repo/results/",
    "joint_language_table.csv":        "Quran folder, or anon_repo/data/metadata/",
    "fertility_by_translation.csv":    "anon_repo/results/translator_variance/, or paper/newresults/translator/",
    "retrieval_pairs.csv":             "Quran folder, or anon_repo/results/",
}

def ask_for(name, where):
    # open an upload dialog for one file; accept it even if the browser renamed it
    from google.colab import files
    while not os.path.exists(name):
        print(f"\nUpload  {name}   (found in: {where})")
        up = files.upload()
        if not up:
            print("  nothing uploaded, asking again"); continue
        got = list(up)[0]
        stem, ext = os.path.splitext(name)
        if got != name and got.startswith(stem) and got.endswith(ext):   # e.g. "retrieval_pairs (1).csv"
            os.replace(got, name)
        elif got != name:
            print(f"  got {got}, expected {name}; ignoring it and asking again")
            os.remove(got); continue
        print(f"  ok: {name} ({os.path.getsize(name):,} bytes)")

try:
    import google.colab  # noqa: F401
    for name, where in NEED.items():
        if os.path.exists(name):
            print(f"already here: {name}")
        else:
            ask_for(name, where)
except ImportError:
    pass
for f in NEED:
    assert os.path.exists(f), f"missing {f}"

try:
    import torch; GPU = torch.cuda.is_available()
except Exception:
    GPU = False
print("GPU:", GPU, "(Part D runs only with a GPU)")

## 1. eBible: candidates for each corpus language

Candidate translations are every eBible entry with a matching ISO 639-3 code that is marked
redistributable. Script is read from the text itself, because the metadata's script column is not
reliable (for example, one Romanian translation is in Cyrillic and one Urdu translation in Devanagari).

In [ ]:
FERT = pd.read_csv("fertility_production_v3_spm.csv")
JOINT = pd.read_csv("joint_language_table.csv")
JOINT = JOINT[JOINT.language != "bosnian-mihanovich"].copy()          # 40 languages, as in the paper
JOINT["lang"] = JOINT.language.replace({"bosnian-rwwad": "bosnian"})
SCRIPT = dict(zip(JOINT.lang, JOINT.script))
TIER = dict(zip(JOINT.lang, JOINT.joshi))
assert len(SCRIPT) == 40

# corpus language -> ISO 639-3 codes that eBible might use for it
ISO = {"albanian": ["sqi", "als", "aln"], "asante": ["twi", "aka"], "assamese": ["asm"],
       "azeri": ["azj", "aze", "azb"], "bosnian": ["bos"], "chinese": ["cmn", "zho"], "croatian": ["hrv"],
       "dutch": ["nld"], "english": ["eng"], "french": ["fra"], "german": ["deu"], "gujarati": ["guj"],
       "hausa": ["hau"], "indonesian": ["ind"], "japanese": ["jpn"], "kannada": ["kan"], "khmer": ["khm"],
       "kurdish": ["ckb"], "kyrgyz": ["kir"], "lingala": ["lin"], "lithuanian": ["lit"],
       "macedonian": ["mkd"], "malayalam": ["mal"], "moore": ["mos"], "oromo": ["gaz", "orm", "hae"],
       "pashto": ["pbt", "pus", "pbu"], "persian": ["pes", "fas", "prs"], "portuguese": ["por"],
       "romanian": ["ron"], "serbian": ["srp"], "spanish": ["spa"], "tagalog": ["tgl", "fil"],
       "tajik": ["tgk"], "tamil": ["tam"], "turkish": ["tur"], "urdu": ["urd"], "uyghur": ["uig"],
       "uzbek": ["uzn", "uzb", "uzs"], "vietnamese": ["vie"], "yoruba": ["yor"]}
assert set(ISO) == set(SCRIPT), set(ISO) ^ set(SCRIPT)

os.makedirs("ebible", exist_ok=True)
def fetch(rel):
    fp = os.path.join("ebible", rel.replace("/", "__"))
    if not os.path.exists(fp):
        try:
            urllib.request.urlretrieve(f"{EBIBLE}/{rel}", fp)
        except urllib.error.HTTPError as e:
            if e.code == 404: return None
            raise
        except urllib.error.URLError as e:          # local mirror (file://) without that file
            if isinstance(e.reason, FileNotFoundError): return None
            raise
    return fp

VREF = [l.rstrip("\n") for l in open(fetch("metadata/vref.txt"), encoding="utf-8")]
META = pd.read_csv(fetch("metadata/translations.csv"))
N_OT = next(i for i, v in enumerate(VREF) if v.startswith("MAT "))
N_BIBLE = next(i for i, v in enumerate(VREF) if v.startswith("TOB "))      # OT + NT, no deuterocanon
print(f"{len(VREF)} verse references; OT {N_OT}, NT {N_BIBLE - N_OT}")

CAT = {"LATIN": "Latin", "ARABIC": "Arabic", "CYRILLIC": "Cyrillic", "CJK": "CJK", "HIRAGANA": "CJK",
       "KATAKANA": "CJK", "HANGUL": "CJK", "KHMER": "Khmer", "BENGALI": "Brahmic", "TAMIL": "Brahmic",
       "MALAYALAM": "Brahmic", "KANNADA": "Brahmic", "GUJARATI": "Brahmic", "DEVANAGARI": "Devanagari",
       "TELUGU": "Brahmic", "ORIYA": "Brahmic", "GURMUKHI": "Brahmic", "SINHALA": "Brahmic"}
def script_of(lines):
    k = Counter()
    for t in lines[::max(1, len(lines) // 400)]:
        for ch in t:
            if ch.isalpha(): k[CAT.get(unicodedata.name(ch, "?").split()[0], "other")] += 1
    return k.most_common(1)[0][0] if k else "none"

def usable(lines):
    # indices of verses that are present, not a <range> marker, and not merged with the next verse
    ok = set()
    for i in range(N_BIBLE):
        t = lines[i].strip()
        if t and t != "<range>" and not (i + 1 < len(lines) and lines[i + 1].strip() == "<range>"):
            ok.add(i)
    return ok

TEXTS, CAND = {}, []
for lang, codes in ISO.items():
    m = META[META.languageCode.isin(codes) & (META.Redistributable == True)]
    for _, r in m.iterrows():
        fid = f"{r.languageCode}-{r.translationId}"
        fp = fetch(f"corpus/{fid}.txt")
        if fp is None:
            CAND.append({"language": lang, "file": fid, "status": "listed, not in public corpus"}); continue
        lines = [l.rstrip("\n") for l in open(fp, encoding="utf-8")]
        assert len(lines) == len(VREF), (fid, len(lines))
        u = usable(lines)
        sc = script_of([lines[i] for i in sorted(u)])
        TEXTS[fid] = (lines, u)
        yr = re.findall(r"(1[5-9]\d\d|20\d\d)", f"{r.translationId} {r.Copyright}")
        CAND.append({"language": lang, "file": fid, "status": "available", "script": sc,
                     "corpus_script": SCRIPT[lang], "script_match": sc == SCRIPT[lang],
                     "usable_verses": len(u), "OT": sum(i < N_OT for i in u), "NT": sum(i >= N_OT for i in u),
                     "year": int(min(yr)) if yr else None,
                     "copyright": str(r.Copyright)[:80], "variety": r.languageNameInEnglish})
CAND = pd.DataFrame(CAND)
CAND.to_csv(f"{OUT}/candidates.csv", index=False)
print(f"{len(CAND)} redistributable eBible entries for the 40 languages; "
      f"{(CAND.status == 'available').sum()} have a public text file")
print(CAND[CAND.status == "available"].drop(columns=["copyright"]).to_string(index=False))

## 2. A — coverage and the shared verse set

Selection rule, fixed before looking at any tokenizer or encoder: among a language's available
translations in the corpus's script, take the one with the most usable verses (ties: alphabetical).
One exception: for Twi, the Asante edition, because the corpus's Twi is Asante.

In [ ]:
av = CAND[CAND.status == "available"].copy(); av["script_match"] = av.script_match.astype(bool)
# the corpus's Twi is Asante Twi; eBible has an Asante edition next to a general "Twi" one
PREFER = {"asante": "twi-twiasante"}
PICK, COVER = {}, []
for lang in sorted(ISO):
    a = av[av.language == lang]
    ok = a[a.script_match].sort_values(["usable_verses", "file"], ascending=[False, True])
    if lang in PREFER and PREFER[lang] in set(ok.file):
        ok = pd.concat([ok[ok.file == PREFER[lang]], ok[ok.file != PREFER[lang]]])
    if len(ok):
        r = ok.iloc[0]; PICK[lang] = r.file
        nt_full = r.NT >= 0.9 * (N_BIBLE - N_OT)
        kind = ("full Bible" if r.OT >= 0.8 * N_OT and nt_full else
                "NT only" if r.OT < 0.05 * N_OT and nt_full else "partial")
        COVER.append({"language": lang, "status": "covered", "file": r.file, "extent": kind,
                      "usable_verses": int(r.usable_verses), "year": r.year, "copyright": r.copyright})
    elif len(a):
        COVER.append({"language": lang, "status": "script mismatch",
                      "file": ", ".join(f"{f} ({s})" for f, s in zip(a.file, a.script))})
    else:
        COVER.append({"language": lang, "status": "absent"})
COVER = pd.DataFrame(COVER)
COVER["corpus_script"] = COVER.language.map(SCRIPT); COVER["joshi"] = COVER.language.map(TIER)
COVER.to_csv(f"{OUT}/A_coverage.csv", index=False)
print(COVER.drop(columns=["copyright"]).to_string(index=False))

LANGS = sorted(PICK)                                   # covered, script-matched languages
NONENG = [l for l in LANGS if l != "english"]
SHARED = sorted(set.intersection(*[TEXTS[PICK[l]][1] for l in LANGS]))
books = Counter(VREF[i].split()[0] for i in SHARED)
# what the shared set would be without the smallest translations
by_size = sorted(LANGS, key=lambda l: len(TEXTS[PICK[l]][1]))
drop_curve = []
for k in range(0, 5):
    keep = [l for l in LANGS if l not in by_size[:k]]
    drop_curve.append({"dropped": ", ".join(by_size[:k]) or "-", "languages": len(keep),
                       "shared_verses": len(set.intersection(*[TEXTS[PICK[l]][1] for l in keep]))})
DROP = pd.DataFrame(drop_curve); DROP.to_csv(f"{OUT}/A_shared_verses_vs_languages.csv", index=False)

holders = COVER.dropna(subset=["copyright"]).copyright.str.replace(r"(?i)copyright|©|\d{4}(,\s*\d{4})*|-", "", regex=True).str.strip()
A = {"languages_covered": len(LANGS),
     "languages_covered_names": LANGS,
     "script_mismatch": COVER[COVER.status == "script mismatch"].language.tolist(),
     "absent": COVER[COVER.status == "absent"].language.tolist(),
     "extent": COVER[COVER.status == "covered"].extent.value_counts().to_dict(),
     "shared_verses_all_covered": len(SHARED),
     "shared_verses_books": dict(books.most_common()),
     "shared_verses_vs_languages": DROP.to_dict(orient="records"),
     "distinct_copyright_holders": int(holders[holders.str.lower() != "public domain"].nunique()),
     "public_domain": int((holders.str.lower() == "public domain").sum()),
     "year_range": [int(COVER.year.min()), int(COVER.year.max())] if COVER.year.notna().any() else None,
     "absent_joshi_0_1": [l for l in COVER[COVER.status != "covered"].language if TIER[l] <= 1],
     "corpus": {"languages": 40, "shared_verses": 6236, "missing_cells": 0, "publishing_projects": 1}}
print(json.dumps({k: v for k, v in A.items() if k != "shared_verses_books"}, indent=1, default=str))
print(f"\n{len(LANGS)} of 40 languages covered; {len(SHARED)} verses are usable in all of them "
      f"(mostly {', '.join(b for b, _ in books.most_common(4))}); the corpus has 6,236 in all 40.")

## 3. B — the tokenizer audit on shared Bible verses

Same eleven tokenizers as the paper (ten distinct vocabularies; Aya-101 ships mT5's). mT5 and XLM-R
are counted with the reference `sentencepiece` library on each model's own model file, the method
behind the corrected counts; the Hugging Face tokenizers are checked against it on a sample. Every
language is scored on the same shared verses.

In [ ]:
import tiktoken
from transformers import AutoTokenizer
from huggingface_hub import hf_hub_download
import sentencepiece as spm

# every counter takes a list of strings and returns a list of token counts
TOK = {}
o200k = tiktoken.get_encoding("o200k_base"); cl100k = tiktoken.get_encoding("cl100k_base")
TOK["gpt4o_o200k"] = lambda ts: [len(x) for x in o200k.encode_batch(ts, disallowed_special=())]
TOK["gpt4_cl100k"] = lambda ts: [len(x) for x in cl100k.encode_batch(ts, disallowed_special=())]
HF = {"mbert": ["bert-base-multilingual-cased"], "bloom": ["bigscience/bloom-560m"],
      "nllb": ["facebook/nllb-200-distilled-600M"],
      "llama3": ["unsloth/llama-3-8b", "Xenova/llama-3-tokenizer"], "gemma": ["unsloth/gemma-2-2b", "unsloth/gemma-2-9b"]}
for key, names in HF.items():
    for n in names:
        try:
            t = AutoTokenizer.from_pretrained(n)
            TOK[key] = (lambda tt: (lambda ts: [len(x) for x in tt(ts, add_special_tokens=False)["input_ids"]]))(t)
            print(f"loaded {key} <- {n}"); break
        except Exception as e:
            print(f"  {key}: {n} failed ({type(e).__name__})")
SPM = {"mt5": ("google/mt5-small", "spiece.model"), "xlmr": ("xlm-roberta-base", "sentencepiece.bpe.model")}
HF_CHECK = {}
for key, (repo, fname) in SPM.items():
    sp = spm.SentencePieceProcessor(model_file=hf_hub_download(repo, fname))
    TOK[key] = (lambda p: (lambda ts: [len(x) for x in p.encode(ts)]))(sp)
    try:
        HF_CHECK[key] = AutoTokenizer.from_pretrained(repo)
    except Exception as e:
        print(f"  (no HF {key} for the cross-check: {type(e).__name__})")
TOK["byt5"] = lambda ts: [len(s.encode("utf-8")) for s in ts]
KEYS = ["gpt4o_o200k", "gpt4_cl100k", "mbert", "xlmr", "bloom", "nllb", "mt5", "llama3", "gemma", "byt5"]
missing = [k for k in KEYS if k not in TOK]
assert not missing, f"tokenizers not loaded: {missing}"
LEARNED = [k for k in KEYS if k != "byt5"]            # nine learned vocabularies

# sentencepiece vs Hugging Face on 200 shared verses per language
chk = []
for key, hft in HF_CHECK.items():
    diff = n = 0
    for l in LANGS:
        vs = [TEXTS[PICK[l]][0][i] for i in SHARED[:200]]
        hf_counts = [len(x) for x in hft(vs, add_special_tokens=False)["input_ids"]]
        diff += sum(int(a != b) for a, b in zip(TOK[key](vs), hf_counts)); n += len(vs)
    chk.append({"tokenizer": key, "verses": n, "differ": diff})
print("sentencepiece vs HF:", chk or "not run")

def bible_fertility(langs, verses):
    rows = []
    for l in langs:
        vs = [TEXTS[PICK[l]][0][i] for i in verses]
        r = {"language": l, "file": PICK[l], "verses": len(vs),
             "bytes_per_verse": np.mean([len(s.encode("utf-8")) for s in vs])}
        for k in KEYS:
            r[f"{k}_tok_per_verse"] = float(np.mean(TOK[k](vs)))
        rows.append(r)
    return pd.DataFrame(rows).set_index("language")

t0 = time.time()
BF = bible_fertility(LANGS, SHARED)
BF.to_csv(f"{OUT}/B_bible_fertility.csv")
print(f"tokenized {len(LANGS)} languages x {len(SHARED)} verses x {len(KEYS)} tokenizers in {time.time()-t0:.0f}s")

The comparison below uses the corpus's counts for the same languages. It is run twice: on all covered
languages (the shared verses are then mostly the Gospels), and without the two partial translations,
which gives a larger and more varied verse set.

In [ ]:
CF = FERT.copy(); CF["lang"] = CF.language.replace({"bosnian-rwwad": "bosnian"})
CF = CF.set_index("lang")

def p90p10(v): v = np.asarray(v, float); return np.percentile(v, 90) / np.percentile(v, 10)
rng = np.random.default_rng(42)
def boot_ci(v):
    v = np.asarray(v, float); b = [p90p10(rng.choice(v, len(v))) for _ in range(N_BOOT)]
    return np.percentile(b, [2.5, 97.5])

def compare(bf, langs, label):
    c_ = CF.loc[langs]; b_ = bf.loc[langs]
    LAT = [l for l in langs if SCRIPT[l] == "Latin"]; BRA = [l for l in langs if SCRIPT[l] == "Brahmic"]
    rows = []
    for k in KEYS:
        col = f"{k}_tok_per_verse"
        c, b = c_[col], b_[col]
        rho, p = stats.spearmanr(c, b)
        ctpb = c / c_.bytes_per_verse; btpb = b / b_.bytes_per_verse
        lo, hi = boot_ci(b)
        rc = np.median(ctpb[BRA]) / np.median(ctpb[LAT]); rb = np.median(btpb[BRA]) / np.median(btpb[LAT])
        rows.append({"set": label, "tokenizer": k, "n": len(c), "rho_language_fertility": round(rho, 3), "p": f"{p:.1e}",
                     "rho_tokens_per_byte": round(stats.spearmanr(ctpb, btpb)[0], 3) if k != "byt5" else None,
                     "corpus_p90p10": round(p90p10(c), 2), "bible_p90p10": round(p90p10(b), 2),
                     "bible_p90p10_lo": round(lo, 2), "bible_p90p10_hi": round(hi, 2),
                     "corpus_brahmic_latin_tpb": round(rc, 2), "bible_brahmic_latin_tpb": round(rb, 2),
                     "bible_brahmic_latin_p": (round(stats.mannwhitneyu(np.log(btpb[BRA]), np.log(btpb[LAT]))[1], 5)
                                               if k != "byt5" else None),
                     "same_direction": bool(np.sign(np.log(rc)) == np.sign(np.log(rb))) if k != "byt5" else None})
    bt = pd.DataFrame(rows)
    rr, rp_ = stats.spearmanr(bt.corpus_p90p10, bt.bible_p90p10)
    L_ = bt[bt.tokenizer.isin(LEARNED)]
    summ = {"set": label, "n_languages": len(langs), "shared_verses": int(b_.verses.iloc[0]),
            "brahmic": BRA, "n_latin": len(LAT),
            "rho_language_fertility_median": float(bt.rho_language_fertility.median()),
            "rho_language_fertility_range": [float(bt.rho_language_fertility.min()), float(bt.rho_language_fertility.max())],
            "tokenizers_rho_ge_0.7": int((bt.rho_language_fertility >= 0.7).sum()),
            "p90p10_rank_agreement_rho": round(float(rr), 3), "p90p10_rank_agreement_p": float(rp_),
            "brahmic_cheaper_corpus": int((L_.corpus_brahmic_latin_tpb < 1).sum()),
            "brahmic_cheaper_bible": int((L_.bible_brahmic_latin_tpb < 1).sum()),
            "direction_agrees": int(L_.same_direction.sum()), "n_learned": len(L_)}
    print(f"\n=== {label}: {len(langs)} languages, {summ['shared_verses']} verses")
    print(bt.drop(columns=["set"]).to_string(index=False))
    print(f"language fertility, Bible vs corpus: median rho {summ['rho_language_fertility_median']:.2f} "
          f"(range {summ['rho_language_fertility_range'][0]:.2f} to {summ['rho_language_fertility_range'][1]:.2f}); "
          f"{summ['tokenizers_rho_ge_0.7']}/10 tokenizers at rho >= 0.7")
    print(f"tokenizer inequality (p90/p10) rank agreement: rho {rr:.2f} (p {rp_:.1e})")
    print(f"Brahmic cheaper than Latin per byte: corpus {summ['brahmic_cheaper_corpus']}/9, "
          f"Bible {summ['brahmic_cheaper_bible']}/9; same direction for {summ['direction_agrees']}/9")
    return bt, summ

BT, B = compare(BF, LANGS, "all covered languages")
PARTIAL = COVER[(COVER.status == "covered") & (COVER.extent == "partial")].language.tolist()
LANGS2 = [l for l in LANGS if l not in PARTIAL]
SHARED2 = sorted(set.intersection(*[TEXTS[PICK[l]][1] for l in LANGS2]))
BF2 = bible_fertility(LANGS2, SHARED2); BF2.to_csv(f"{OUT}/B_bible_fertility_without_partial.csv")
BT2, B2 = compare(BF2, LANGS2, f"without {', '.join(PARTIAL)}")
pd.concat([BT, BT2]).to_csv(f"{OUT}/B_tokenizer_comparison.csv", index=False)
B["sentencepiece_vs_hf"] = chk
B["sensitivity"] = B2

## 4. C — translator variance

The corpus's translations come from one publishing project. eBible's come from many publishers across
more than a century. Here, for every language with two or more usable translations in the corpus's
script (at least 4,000 usable verses, near-identical editions removed), fertility is compared across
them on the verses they share, and set against the paper's
translator study (33 translations of 14 languages). The median ratio over all pairs of translations is
reported because the maximum grows with the number of translations.

In [ ]:
def pair_ratios(vals):
    return [max(a, b) / min(a, b) for a, b in itertools.combinations(vals, 2)]

def distinct_editions(files):
    # drop editions that repeat another kept edition's text (e.g. the World English Bible variants):
    # an edition is dropped if half or more of the verses both have are identical to a kept one
    kept = []
    for f in files:
        lf, uf = TEXTS[f]
        dup = False
        for g in kept:
            lg, ug = TEXTS[g]; both = uf & ug
            if both and sum(lf[i] == lg[i] for i in both) >= 0.5 * len(both):
                dup = True; break
        if not dup: kept.append(f)
    return kept

rows, brows, DUPS = [], [], {}
for lang in sorted(ISO):
    a = av[(av.language == lang) & av.script_match & (av.usable_verses >= 4000)].sort_values("usable_verses", ascending=False)
    files = distinct_editions(list(a.file))
    if len(files) < len(a): DUPS[lang] = sorted(set(a.file) - set(files))
    if len(files) < 2: continue
    common = sorted(set.intersection(*[TEXTS[f][1] for f in files]))
    for f in files:
        vs = [TEXTS[f][0][i] for i in common]
        brows.append({"language": lang, "file": f, "verses": len(common),
                      "bytes_per_verse": np.mean([len(s.encode("utf-8")) for s in vs]),
                      "gpt4o_o200k_tpv": float(np.mean(TOK["gpt4o_o200k"](vs)))})
BV = pd.DataFrame(brows); BV.to_csv(f"{OUT}/C_bible_by_translation.csv", index=False)
print("editions left out as near-copies of another:", DUPS)

CV = pd.read_csv("fertility_by_translation.csv")
def spread(df, label):
    out = []
    for lang, g in df.groupby("language"):
        for m in ["bytes_per_verse", "gpt4o_o200k_tpv"]:
            v = g[m].values
            out.append({"source": label, "language": lang, "measure": m, "translations": len(v),
                        "max_over_min": v.max() / v.min(), "median_pair_ratio": np.median(pair_ratios(v))})
    return out
SPR = pd.DataFrame(spread(BV, "bible") + spread(CV, "corpus study"))
SPR.to_csv(f"{OUT}/C_within_language_spread.csv", index=False)
C_ = (SPR.groupby(["source", "measure"])
         .agg(languages=("language", "nunique"), translations=("translations", "sum"),
              median_max_over_min=("max_over_min", "median"), median_pair_ratio=("median_pair_ratio", "median"),
              worst=("max_over_min", "max")).round(3))
print(C_.to_string())
both = sorted(set(BV.language) & set(CV.language))
print(f"\nlanguages in both studies: {both}")
print(SPR[SPR.language.isin(both) & (SPR.measure == "gpt4o_o200k_tpv")]
      .pivot(index="language", columns="source", values="median_pair_ratio").round(3).to_string())
Cs = {"summary": C_.reset_index().to_dict(orient="records"), "languages_in_both": both,
      "near_copies_dropped": DUPS}

## 5. D — retrieval on shared verses (GPU)

English-paired retrieval in both directions, top-1 by cosine over normalized embeddings, averaged over
the two directions, on 1,000 of the shared verses (seed 0), exactly as the Tatoeba comparison did. The
corpus side is read from the paper's `retrieval_pairs.csv` (English rows), so it is the paper's own
number for each language.

In [ ]:
ENCODERS = {"labse": "sentence-transformers/LaBSE", "bgem3": "BAAI/bge-m3",
            "e5": "intfloat/multilingual-e5-base",
            "mpnet": "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"}
D = None
if GPU:
    import torch
    from sentence_transformers import SentenceTransformer
    rng0 = np.random.default_rng(0)
    SAMPLE = [SHARED[i] for i in rng0.choice(len(SHARED), size=min(1000, len(SHARED)), replace=False)]
    def top1(A, Bm):
        S = A @ Bm.T; n = len(A)
        return (S.argmax(1) == np.arange(n)).mean(), (S.argmax(0) == np.arange(n)).mean()
    rows = []; t0 = time.time()
    for tag, name in ENCODERS.items():
        model = SentenceTransformer(name, device="cuda")
        pre = "query: " if "e5" in name else ""
        emb = lambda ts: model.encode([pre + t for t in ts], batch_size=64, convert_to_numpy=True,
                                      normalize_embeddings=True, show_progress_bar=False)
        E = emb([TEXTS[PICK["english"]][0][i] for i in SAMPLE])
        for l in NONENG:
            X = emb([TEXTS[PICK[l]][0][i] for i in SAMPLE])
            a, b = top1(X, E)
            rows.append({"encoder": tag, "language": l, "bible_x2en": a, "bible_en2x": b, "bible": (a + b) / 2})
        print(f"  {tag:6s} done ({time.time()-t0:.0f}s)", flush=True)
        del model; torch.cuda.empty_cache()
    RB = pd.DataFrame(rows)

    rp = pd.read_csv("retrieval_pairs.csv"); TAG = {v: k for k, v in ENCODERS.items()}
    rp["encoder"] = rp.model.map(TAG)
    rp["src"] = rp.src.replace({"bosnian-rwwad": "bosnian"}); rp["tgt"] = rp.tgt.replace({"bosnian-rwwad": "bosnian"})
    def corpus_eng(enc, l):
        a = rp[(rp.encoder == enc) & (rp.src == l) & (rp.tgt == "english")].top1
        b = rp[(rp.encoder == enc) & (rp.src == "english") & (rp.tgt == l)].top1
        return (a.iloc[0] + b.iloc[0]) / 2
    RB["corpus"] = [corpus_eng(e, l) for e, l in zip(RB.encoder, RB.language)]
    RB["bible_minus_corpus"] = RB.bible - RB.corpus
    RB.to_csv(f"{OUT}/D_retrieval_by_language.csv", index=False)

    K = 5; rows = []
    for enc in ENCODERS:
        d = RB[RB.encoder == enc].set_index("language")
        rho, p = stats.spearmanr(d.corpus, d.bible)
        bc = set(d.corpus.nsmallest(K).index); bb = set(d.bible.nsmallest(K).index)
        rows.append({"encoder": enc, "n": len(d), "rho": round(rho, 3), "p": round(p, 4),
                     "corpus_mean": round(d.corpus.mean(), 3), "bible_mean": round(d.bible.mean(), 3),
                     "corpus_bottom5": ", ".join(d.corpus.nsmallest(K).index),
                     "bible_bottom5": ", ".join(d.bible.nsmallest(K).index),
                     "bottom5_overlap": len(bc & bb)})
    DR = pd.DataFrame(rows); DR.to_csv(f"{OUT}/D_retrieval_agreement.csv", index=False)
    print(DR.to_string(index=False))
    FLOOR = ["oromo", "asante", "lingala", "yoruba", "kurdish", "hausa"]
    fl = RB[RB.language.isin(FLOOR)].pivot(index="language", columns="encoder", values=["corpus", "bible"]).round(3)
    print("\nlow-resource African and Kurdish languages (Moore is not in eBible):\n", fl.to_string())
    gain = RB.assign(floor=RB.language.isin(FLOOR)).groupby(["encoder", "floor"]).bible_minus_corpus.median().unstack().round(3)
    print("\nmedian Bible minus corpus accuracy, floor languages vs the rest:\n", gain.to_string())
    D = {"agreement": DR.to_dict(orient="records"), "floor_gain": gain.reset_index().to_dict(orient="records"),
         "sample": len(SAMPLE)}
else:
    print("No GPU: Part D skipped. Runtime -> Change runtime type -> T4, then run this cell again.")

## Summary and download

In [ ]:
S = {"A_coverage": A, "B_tokenizers": B, "B_table": pd.concat([BT, BT2]).to_dict(orient="records"),
     "C_translator_variance": Cs, "D_retrieval": D}
json.dump(S, open(f"{OUT}/summary.json", "w"), indent=1, default=str)
print(json.dumps({"A": {k: A[k] for k in ["languages_covered", "absent", "script_mismatch", "extent",
                                          "shared_verses_all_covered", "distinct_copyright_holders", "year_range"]},
                  "B": {k: B[k] for k in ["rho_language_fertility_median", "p90p10_rank_agreement_rho",
                                          "brahmic_cheaper_corpus", "brahmic_cheaper_bible", "direction_agrees"]}},
                 indent=1, default=str))
import shutil; shutil.make_archive("results_bible", "zip", OUT)
try:
    from google.colab import files; files.download("results_bible.zip")
except Exception:
    print("written results_bible.zip")

## Reading the results

Decided before running, and reported whichever way they come out.

| Result | Meaning |
|---|---|
| **A** eBible lacks several corpus languages, and the verses shared by all covered languages are far fewer than 6,236 | For this language set, the Bible cannot give a complete, gap-free parallel design; the study would have lost languages or verses. Which languages are lost matters most: if they are low-resource ones, the Bible drops exactly the languages the paper is about. |
| **B** ρ ≥ 0.7 for most tokenizers, p90/p10 rank agreement ≥ 0.7, and the Brahmic/Latin direction agrees for the nine learned vocabularies | The tokenizer findings are not specific to the Qur'an: they reproduce on an independent religious corpus with different translators. |
| **B** weak agreement, or the direction flips | Either domain or translation choices drive part of the audit; this must be reported and the claims narrowed. |
| **C** Bible within-language spread larger than the corpus's | The single-project design reduces translator variance, which is a measured advantage of the corpus, not an assumed one. |
| **C** similar spread | No advantage on this axis; the corpus's advantage is coverage and completeness. |
| **D** ρ high and the same languages at the bottom | The retrieval floor is not specific to the Qur'an. |
| **D** floor languages gain much more on the Bible than other languages | Consistent with Bible text in the encoders' training data, since Bible translations are a main source of parallel text for low-resource languages; domain differences cannot be separated from this here, and it should be stated. |